In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
import MDAnalysis as mda
import numpy as np
import subprocess
import matplotlib.pyplot as plt
import chemtrain
from jax_md import space

In [ ]:
u = mda.Universe("heavy_first.gro", "heavyMD.trr")
print(f"Number of atoms detected: {len(u.atoms)}")

In [ ]:
for atom in u.atoms:
    print(f"Atom ID: {atom.id}, Name: {atom.name}, Type: {atom.type}, Mass: {atom.mass}")

In [ ]:
# MDA units:
# length: Angstrom
# velocity: Angstrom/ps
# Forces : kJ/mol/A

# Gromacs units:
# length: nm = A/10
# velocity: nm/ps = A/ps/10
# Forces : kJ/mol/nm = 10 * kJ/mol/A 

# Prepare containers for coordinates and forces
data = {
    res.resid: {"resname": res.resname, "coords": [], "forces": []}
    for res in u.residues
}

# Loop over frames to collect positions and forces
for ts in u.trajectory:
    positions = ts.positions
    forces = getattr(ts, "forces", None)
    for res in u.residues:
        idx = res.atoms.indices
        data[res.resid]["coords"].append(positions[idx])
        if forces is not None:
            data[res.resid]["forces"].append(forces[idx])

print("Collected coordinates and forces for all molecules across frames.")

In [ ]:
BOX_LENGTH = 3.68276
species = [1,1,2,3,1,1,1,2,3,1]
import jax.numpy as jnp
box = jnp.identity(3) * BOX_LENGTH

displacement_fn, _ = space.periodic_general(box=box, fractional_coordinates=False)

In [ ]:
# Initialize dataset dictionary if it doesn't exist
dataset = {}

n_frames_list = [len(res["coords"]) for res in data.values()]
assert len(set(n_frames_list)) == 1, "Mismatch in number of frames per residue"

# Get number of frames and total atoms
if data and len(data) > 0:
    first_resid = next(iter(data))
    num_frames = len(data[first_resid]["coords"])
    n_atoms = sum(len(data[resid]["coords"][0]) for resid in data if data[resid]["coords"])
    
    assert len(species) == n_atoms, (
        f"species length ({len(species)}) != total atoms ({n_atoms})"
    )
    
    print(f"Processing {num_frames} frames with {n_atoms} total atoms")
    
    # 2) Count atoms
    n_atoms = sum(len(d["coords"][0]) for d in data.values())
    assert len(species) == n_atoms, "Species length ≠ atom count"

    # 3) Preallocate
    R = np.zeros((num_frames, n_atoms, 3), np.float32)
    F = np.zeros_like(R)

    # 4) Fill arrays
    ordered_resids = sorted(data.keys())  # or your canonical order
    for t in range(num_frames):
        off = 0
        for resid in ordered_resids:
            coords = data[resid]["coords"][t]
            forces = data[resid]["forces"][t]
            nat = coords.shape[0]
            R[t, off:off+nat] = coords
            F[t, off:off+nat] = forces
            off += nat
        
    R *= 0.1  # Å → nm
    F *= 10 # kcal/mol/Å → kJ/mol/nm
    
    dataset['R'] = R
    dataset['F'] = F
    
    n_frames = R.shape[0]
    box = np.identity(3) * BOX_LENGTH # nm
    
    dataset['box'] = np.repeat(box.reshape(1, 3, 3), n_frames, axis=0)
    dataset['species'] = np.repeat(np.array(species).reshape(1, len(species)), n_frames, axis=0)
    dataset['mask'] = np.repeat(np.array([True]*len(species)).reshape(1, len(species)), n_frames, axis=0)
    
    print(f"Dataset created with shapes: R={R.shape}, F={F.shape}")
else:
    print("No data available to create dataset")

# # save as npz
name = "37a/unbiased_05fs.npz"
np.savez_compressed(name, **dataset)
print(f"Dataset saved as {name}")

In [ ]:
for key, value in dataset.items():
    print(f"{key}: {value.shape}")